# WESAD — KD Hyperparameter Ablation (Session 3)

**Before running:**
1. Notebook Settings -> Accelerator -> **GPU T4 x2** (or P100).
2. Notebook Settings -> Internet -> **On**.
3. Add Data -> attach the same WESAD dataset used in Sessions 1 & 2.
4. Add Data -> attach the **teacher checkpoints** (same as Session 2 — Session 1's
   output, or a dataset of the 15 `teacher_loso_S*.pt` files).

Sweeps KD temperature (4 values) and alpha (4 values) on MicroCNN — each value is
a full 15-fold LOSO training run, so the full sweep (8 x 15 = 120 fold-trainings)
is roughly the same size as the entire student training sweep from Session 2.

If it doesn't fit in one 12h session, use `SWEEP` below to split temperature and
alpha into two separate runs — no code changes needed, `run_ablation.py` already
supports this via `--sweep`.

In [ ]:
import os, glob

# Auto-detect the attached WESAD dataset by locating the S2/S2.pkl marker
# file anywhere under /kaggle/input. Recursive so it doesn't matter how the
# dataset was packaged (some mirrors add an extra top-level 'WESAD' folder).
candidates = glob.glob('/kaggle/input/**/S2/S2.pkl', recursive=True)
if not candidates:
    print('No match. Top-level /kaggle/input contents:',
          os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else '(missing)')
assert candidates, (
    "WESAD dataset not found under /kaggle/input. "
    "Attach it via 'Add Data' first (must contain S2/S2.pkl ... S17/S17.pkl)."
)
data_root = os.path.dirname(os.path.dirname(candidates[0]))
print('Detected WESAD data root:', data_root)

os.environ['WESAD_DATA_DIR'] = data_root
os.environ['WESAD_OUTPUT_DIR'] = '/kaggle/working/outputs'

In [ ]:
REPO_DIR = '/kaggle/working/healthcare_wesad'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/RiverRover-stack/healthcare_wesad.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Bridge the teacher checkpoints into this session's MODELS_DIR — the ablation
# sweep runs distilled KD training, which needs a frozen teacher per fold.
import shutil

os.makedirs('outputs/models', exist_ok=True)
found = glob.glob('/kaggle/input/**/teacher_loso_S*.pt', recursive=True)
assert found, (
    'No teacher checkpoints found under /kaggle/input. '
    "Attach Session 1's output (or a dataset of the 15 teacher_loso_S*.pt files) via 'Add Data'."
)
for f in found:
    shutil.copy(f, 'outputs/models/')

copied = sorted(glob.glob('outputs/models/teacher_loso_S*.pt'))
print(f'Copied {len(copied)} teacher checkpoints into outputs/models/:')
for c in copied:
    print(' ', c)
assert len(copied) == 15, f'Expected 15 teacher checkpoints, found {len(copied)}.'

In [ ]:
# Sanity checks before committing to the full sweep.
import sys
sys.path.insert(0, 'src')

import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not enabled — check Notebook Settings -> Accelerator.'

from data import load_all_subjects
_probe = load_all_subjects(subject_ids=['S2'])
assert 'S2' in _probe, 'Failed to load S2 — check dataset path/structure.'
print('Sanity check passed.')

In [ ]:
# Edit to split the sweep across sessions if 12h isn't enough. Examples:
#   SWEEP = 'temperature'   # temperature sweep only, alpha in a later session
#   SWEEP = 'alpha'         # alpha sweep only
SWEEP = 'both'
MODEL = 'MicroCNN'

cmd = f'python run_ablation.py --sweep {SWEEP} --model {MODEL}'
print('Running:', cmd)
!{cmd}

In [ ]:
# Show the resulting ablation table, then zip everything for download.
with open('/kaggle/working/outputs/reports/ablation_results.csv') as f:
    print(f.read())

!cd /kaggle/working && zip -rq ablation_outputs.zip outputs/models outputs/reports
print('Saved /kaggle/working/ablation_outputs.zip — download it from the notebook Output tab.')